In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

df = pd.read_csv(r"C:\Users\Administrator\Documents\kenya_market_prices\data\food-prices-kenya.csv")

df.head()


: 

In [ ]:
import pandas as pd

prices = pd.read_csv("data/food-prices-kenya.csv")

n_rows = prices.shape[0]
n_markets = prices['market'].nunique()
n_commodities = prices['commodity'].nunique()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.sample(50)

In [ ]:
#turn the date column into real dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["date"].sample(20, random_state=42)


In [ ]:
#add year and month columns
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month

In [ ]:
df.sample(5)

In [ ]:
#fill the missing county and region for the market that has none
#df["county"].isna().sum()
df["region"].isna().sum()





In [ ]:
#How many rows, markets and commodities does the file have, 
df.shape
#2742 -rows, 11- columns

In [ ]:
#and what are the first and last dates?

# Highest and lowest date in the dataset
highest_date = df['date'].max()
lowest_date = df['date'].min()

print(highest_date)
print(lowest_date)

#last date2019-12-15
#first date2006-01-15

In [ ]:
#which market has no county and how many rows does that affect?
nulls = (
    df[df["county"].isna()]          # keep only rows where county is missing
    .groupby("market")
    .size()
    .sort_values(ascending=False)
)
print(nulls)

In [ ]:
df.duplicated().sum()

In [ ]:
# Fill the missing county and region for Hola market
# Hola is located in Tana River county within the Coast region
df.loc[df['market'] == 'Hola', 'county'] = df.loc[df['market'] == 'Hola', 'county'].fillna('Tana River')
df.loc[df['market'] == 'Hola', 'region'] = df.loc[df['market'] == 'Hola', 'region'].fillna('Coast')

In [ ]:
""" Compare like with like
Split the data into wholesale (per 90 KG bag) and retail (per KG) prices and never average them together. 
Add a price_per_kg column so you can compare what a trader pays per KG with what a household pays per KG.
- Which units appear for wholesale maize and for retail maize?
- On average, how does the retail price per KG of maize compare with the wholesale price per KG?
"""

In [ ]:
df["price_per_kg"] = np.where(df["unit"] == "90 KG", df["price_kes"] / 90, df["price_kes"])
df.sample(10)

In [ ]:
#this code splits the dataset into a wholesale-only dataframe
df_wholesale = df[(df["price_type"] == "Wholesale") & (df["unit"] == "90 KG")].copy()
df_wholesale.head()

In [ ]:
# this code splits the dataset into a retail-only dataframe
df_retail = df[(df["price_type"] == "Retail") & (df["unit"] == "KG")].copy()
df_retail.head()

In [ ]:
df_wholesale.loc[df_wholesale["commodity"] == "Maize", "price_per_kg"].mean()


In [ ]:
df_retail.loc[df_retail["commodity"] == "Maize", "price_per_kg"].mean()

In [ ]:
#Calculate the average wholesale maize price per bag for each year and the year-on-year change. Plot the yearly average as a line chart with the price in KES on the y-axis.

df_maize = df[(df["commodity"]== "Maize") & (df["price_type"] == "Wholesale")].copy()
df_maize.head(10)

In [ ]:

#Calculate the average wholesale maize price per bag for each year and the year-on-year change.
yearly_avg = (
    df_maize.groupby("year", as_index=False)["price_kes"]
    .mean()
    .rename(columns={"price_kes": "avg_price_kes"})
)[["year", "avg_price_kes"]]
yearly_avg

In [ ]:

#Which year had the highest average wholesale maize price? 2011
#Which year saw the biggest rise, and which saw the biggest fall, compared with the year before? biggest rise: 2011, biggest fall:2012

yearly_avg["yoy_change_kes"] = yearly_avg["avg_price_kes"].diff()
yearly_avg

In [ ]:
yearly_avg.loc[yearly_avg["avg_price_kes"].idxmax()]

In [ ]:
plt.plot(yearly_avg["year"], yearly_avg["avg_price_kes"], color="red")
plt.xlabel("Year")
plt.ylabel("Average Price (KES)")
plt.title("Average ")
plt.show()

In [ ]:
print(
    f"Highest wholesale price year: "
    f"{yearly_avg.loc[yearly_avg['avg_price_kes'].idxmax(), 'year']}"
)

In [ ]:
print(
    f"Lowest wholesale price year: "
    f"{yearly_avg.loc[yearly_avg['avg_price_kes'].idxmin(), 'year']}"
)

In [ ]:
yearly_avg.loc[yearly_avg["yoy_change_kes"].idxmax()]
#yearly_avg.loc[yearly_avg["yoy_change_kes"].idxmin()]
